# Split YouTube Videos at Custom Timestamps

Downloads a YouTube video at highest quality and splits it into clips at specified timestamps.
Clips are losslessly cut (stream copy) for maximum quality and speed.

In [11]:
!pip install -q yt-dlp

In [ ]:
import re
import subprocess
import os
from pathlib import Path

# ============================================================
# CONFIGURE HERE
# ============================================================

YOUTUBE_URL = "https://www.youtube.com/watch?v=D9drOuHyzVs"

# Clips: list of (start, end) timestamps.
# Use "HH:MM:SS", "MM:SS", or seconds as float.
CLIPS = [
    ("0:15",  "14:58"),
    ("15:16",  "40:01"),
    ("46:51",  "01:00:09"),
    ("01:01:14",  "01:14:23"),
    ("01:14:48",  "01:23:06"),
    ("01:26:50",  "01:38:53"),
    ("01:42:00",  "02:00:02"),
    ("02:00:27",  "02:40:29"),
]

ROOT = Path("../..").resolve()
OUTPUT_DIR = ROOT / "data/cache/segments_for_labeling"
# ============================================================

# Extract video ID
match = re.search(r'(?:v=|youtu\.be/)([a-zA-Z0-9_-]{11})', YOUTUBE_URL)
VIDEO_ID = match.group(1) if match else "unknown"
print(f"Video ID: {VIDEO_ID}")

## 1. Download at Highest Quality

In [13]:
OUTPUT_DIR.mkdir(exist_ok=True)
raw_video = OUTPUT_DIR / "source.mp4"

if raw_video.exists():
    print(f"Already downloaded: {raw_video}")
else:
    !yt-dlp \
        -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/bestvideo+bestaudio/best" \
        --merge-output-format mp4 \
        -o "{raw_video}" \
        "{YOUTUBE_URL}"

# Show video info
!ffprobe -v quiet -show_entries format=duration,size -show_entries stream=codec_name,width,height,r_frame_rate -of compact "{raw_video}"

## 2. Split into Clips

Lossless stream copy (`-c copy`) — no re-encoding, no quality loss, instant.

In [14]:
def split_clip(source, start, end, name, output_dir):
    """Lossless split — stream copy, no re-encoding."""
    out_path = output_dir / f"{name}.mp4"
    if out_path.exists():
        print(f"  [skip] {out_path.name}")
        return out_path

    cmd = [
        "ffmpeg", "-y",
        "-ss", str(start),
        "-to", str(end),
        "-i", str(source),
        "-c", "copy",
        "-avoid_negative_ts", "make_zero",
        "-loglevel", "error",
        str(out_path),
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  [ERROR] {name}: {result.stderr[:200]}")
        return None

    probe = subprocess.run(
        ["ffprobe", "-v", "quiet", "-show_entries", "format=duration",
         "-of", "csv=p=0", str(out_path)],
        capture_output=True, text=True
    )
    dur = float(probe.stdout.strip()) if probe.stdout.strip() else 0
    print(f"  {name}.mp4  ({start} -> {end}, {dur:.1f}s)")
    return out_path

In [15]:
print(f"Splitting {len(CLIPS)} clips from {raw_video}\n")

clip_paths = []
for i, (start, end) in enumerate(CLIPS):
    name = f"{VIDEO_ID}_{i:04d}"
    path = split_clip(raw_video, start, end, name, OUTPUT_DIR)
    if path:
        clip_paths.append(path)

print(f"\nDone. {len(clip_paths)} clips in {OUTPUT_DIR}/")

## 3. Preview Clips

In [16]:
from IPython.display import Video, display

for path in clip_paths[:3]:  # preview first 3
    print(f"\n{path.name}")
    display(Video(str(path), embed=True, width=640))

## 4. List All Clips

In [17]:
!ls -lh {OUTPUT_DIR}/*.mp4